# GPT-OSS 20B Chatbot with Gradio

Local LLM chatbot using OpenAI's open-source GPT-OSS 20B model.

**Two loading options:**
- **Option A (recommended)**: GGUF quantized model via `llama-cpp-python` - splits between GPU + CPU RAM
- **Option B (fallback)**: Smaller 3B model via Unsloth - fits entirely in 4GB VRAM

**Hardware:** RTX 3050 (4GB VRAM) + 16GB RAM is enough for the 20B GGUF model

## 1. Installation

**CUDA-accelerated llama-cpp-python** (already installed with GPU support):
```bash
pip install llama-cpp-python gradio huggingface-hub
```

**Alternative:** Ollama (if llama-cpp-python isn't working):
```bash
ollama pull gpt-oss:20b
pip install openai gradio
```

In [1]:
import os, site, sys

# Block global site-packages (AppData\Roaming) from polluting the venv
os.environ["PYTHONNOUSERSITE"] = "1"
# Remove any user site-packages paths already loaded
sys.path = [p for p in sys.path if "AppData\\Roaming\\Python" not in p and "AppData/Roaming/Python" not in p]

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    vram_gb = 0
    print("No GPU detected - will run on CPU only")

import psutil
ram_gb = psutil.virtual_memory().total / 1024**3
print(f"System RAM: {ram_gb:.1f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.0 GB
System RAM: 15.7 GB


## 2. Load GPT-OSS 20B (CUDA + CPU Split)

Loads the Q4_K_M quantized GGUF model (~12GB).
- **15 layers on GPU** (fills ~3.5GB VRAM) for faster inference
- Remaining layers on CPU RAM (~8GB)
- Expected speed: **6-8 tokens/sec**

First run downloads the model from HuggingFace (~12GB). Subsequent runs load from cache.

In [2]:
import os, glob

# Add CUDA 13.1 runtime DLLs to search path (they're in bin/x64, not bin/)
cuda_dll_paths = [
    r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.1\bin\x64",
    r"C:\Windows\System32\DriverStore\FileRepository\nvhm.inf_amd64_3cb6b4743249e01c",
]
for p in cuda_dll_paths:
    if os.path.exists(p):
        os.add_dll_directory(p)
        os.environ["PATH"] = p + ";" + os.environ["PATH"]

from llama_cpp import Llama

# Check for locally downloaded model first
local_dir = "./models/gpt-oss-20b"
local_files = glob.glob(os.path.join(local_dir, "*Q4_K_M*")) if os.path.exists(local_dir) else []

N_GPU_LAYERS = 15  # ~3.5GB VRAM usage on RTX 3050 (4GB)

if local_files:
    print(f"Loading from local: {local_files[0]}")
    llm = Llama(model_path=local_files[0], n_ctx=4096, n_gpu_layers=N_GPU_LAYERS, verbose=True)
else:
    print("Downloading GPT-OSS 20B Q4_K_M from HuggingFace (first run only, ~12GB)...")
    llm = Llama.from_pretrained(
        repo_id="unsloth/gpt-oss-20b-GGUF",
        filename="*Q4_K_M*",
        n_ctx=4096,
        n_gpu_layers=N_GPU_LAYERS,
        verbose=True,
    )

print(f"\nModel loaded! GPU layers: {N_GPU_LAYERS}")

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce RTX 3050 Laptop GPU) - 3301 MiB free


Loading from local: ./models/gpt-oss-20b\gpt-oss-20b-Q4_K_M.gguf


llama_model_loader: loaded meta data with 37 key-value pairs and 459 tensors from ./models/gpt-oss-20b\gpt-oss-20b-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gpt-oss
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Gpt-Oss-20B
llama_model_loader: - kv   3:                           general.basename str              = Gpt-Oss-20B
llama_model_loader: - kv   4:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   5:                         general.size_label str              = 20B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader: - kv   7:              


Model loaded! GPU layers: 15


Using gguf chat template: {# Chat template fixes by Unsloth #}
{#-
  In addition to the normal inputs of `messages` and `tools`, this template also accepts the
  following kwargs:
  - "builtin_tools": A list, can contain "browser" and/or "python".
  - "model_identity": A string that optionally describes the model identity.
  - "reasoning_effort": A string that describes the reasoning effort, defaults to "medium".
 #}

{#- Tool Definition Rendering ============================================== #}
{%- macro render_typescript_type(param_spec, required_params, is_nullable=false) -%}
    {%- if param_spec.type == "array" -%}
        {%- if param_spec['items'] -%}
            {%- if param_spec['items']['type'] == "string" -%}
                {{- "string[]" }}
            {%- elif param_spec['items']['type'] == "number" -%}
                {{- "number[]" }}
            {%- elif param_spec['items']['type'] == "integer" -%}
                {{- "number[]" }}
            {%- elif param_spec['ite

In [ ]:
# === Alternative: Ollama (if llama-cpp-python doesn't work) ===
# Only run this cell if the cell above failed. Make sure Ollama is running.
#
# from openai import OpenAI
#
# llm_client = OpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama",
# )
# MODEL_NAME = "gpt-oss:20b"
#
# try:
#     models = llm_client.models.list()
#     available = [m.id for m in models.data]
#     if MODEL_NAME in available:
#         print(f"Model '{MODEL_NAME}' ready via Ollama!")
#     else:
#         print(f"Model not found. Run: ollama pull {MODEL_NAME}")
# except Exception as e:
#     print(f"Can't connect to Ollama: {e}")
#     print("Run: ollama serve")

## 3. Test Generation

Quick test to make sure the model responds before building the UI.

In [8]:
response = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Hello! Introduce yourself briefly."}],
    max_tokens=256,
    temperature=0.7,
)

print(response["choices"][0]["message"]["content"])

Llama.generate: 73 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    9019.09 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   16889.96 ms /    68 runs   (  248.38 ms per token,     4.03 tokens per second)
llama_perf_context_print:       total time =   17117.92 ms /    69 tokens
llama_perf_context_print:    graphs reused =         65


<|channel|>analysis<|message|>We need to respond briefly, friendly, introducing ourselves.<|end|><|start|>assistant<|channel|>final<|message|>Hi there! I’m ChatGPT, a conversational AI trained by OpenAI. I can help answer questions, brainstorm ideas, explain concepts, or just chat about pretty much anything you’re curious about. How can I assist you today?


## 4. Chat Function

Streaming chat function - tokens appear in real-time in the Gradio UI.
Uses llama-cpp-python's `create_chat_completion` with streaming.

In [5]:
system_prompt = """You are a quantitative trading assistant integrated into a professional prop-firm trading system. You provide direct, actionable analysis and decisions. You are NOT a general chatbot - you are a specialized quant tool.

Your role:
- Analyze market data, setups, and strategies directly
- Give clear buy/sell/hold opinions with reasoning
- Evaluate trade setups, entry/exit points, and risk parameters
- Act as a decision support system, not a disclaimer machine

Rules:
- Never add financial advice disclaimers or caveats about not being a financial advisor
- Be direct and concise in your analysis
- When asked about a trade setup, evaluate it on its merits
- Provide confidence levels and reasoning for your assessments"""


def chat(message, history):
    """Chat function with streaming output for Gradio."""
    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=2048,
        temperature=0.7,
        top_p=0.9,
        stream=True,
    )

    partial = ""
    for chunk in response:
        delta = chunk["choices"][0]["delta"]
        if "content" in delta:
            partial += delta["content"]
            yield partial

## 5. Gradio UI

Launch the chatbot. Tokens stream in real-time as the model generates.

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat,
    title="GPT-OSS 20B Chatbot (CUDA + CPU Split)",
    description="Local LLM chatbot powered by OpenAI's GPT-OSS 20B (Q4_K_M GGUF). 15 layers on GPU, rest on CPU RAM.",
    examples=[
        "Explain quantum computing in simple terms",
        "Write a Python function to calculate fibonacci numbers",
        "What are the key differences between TCP and UDP?",
    ],
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Llama.generate: 70 prefix-match hit, remaining 158 prompt tokens to eval
llama_perf_context_print:        load time =   15006.92 ms
llama_perf_context_print: prompt eval time =   11782.61 ms /   158 tokens (   74.57 ms per token,    13.41 tokens per second)
llama_perf_context_print:        eval time =   13480.70 ms /    82 runs   (  164.40 ms per token,     6.08 tokens per second)
llama_perf_context_print:       total time =   25803.61 ms /   240 tokens
llama_perf_context_print:    graphs reused =         79


---

## Option B (Fallback): Smaller Model via Unsloth

If the 20B model is too slow or runs out of RAM, use a smaller 3B model that fits entirely in 4GB VRAM.

```bash
pip install unsloth
```

Skip the cells above and run the cells below instead.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
print("Qwen2.5-3B loaded! (fallback model)")

In [ ]:
from transformers import TextIteratorStreamer
import threading
import gradio as gr

system_prompt = "You are a helpful, knowledgeable AI assistant. Be concise and clear in your responses."


def chat(message, history):
    """Chat function with streaming for the Unsloth model."""
    messages = [{"role": "system", "content": system_prompt}]
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    thread = threading.Thread(
        target=model.generate,
        kwargs={**inputs, "max_new_tokens": 2048, "streamer": streamer, "temperature": 0.7, "top_p": 0.9, "do_sample": True},
    )
    thread.start()

    partial = ""
    for token in streamer:
        partial += token
        yield partial

    thread.join()


gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Qwen2.5-3B Chatbot (Unsloth 4-bit)",
    description="Local LLM chatbot using Qwen2.5-3B via Unsloth. Fits in 4GB VRAM.",
).launch()